In [ ]:
%reload_ext autoreload
%autoreload 2
import sys
import os

# Adjust the path to your project root if needed
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import weibull_min
from scipy.optimize import curve_fit
from functions.functions_dp import *

epsilon_def = 3.84
# Read the data from the Excel file
df = pd.read_excel('../data/Gasverbräuche Bensheim_Etagenheizungen_grouped.xlsx')

# Distribuition of the real gas consumptions of 114 buildings in Bensheim

In [ ]:
plot_index(df[2023])

# Weibull Distribution and Gas Consumption Data

The Weibull distribution is defined by its shape parameter  $k$  and scale parameter  $\lambda$ . The probability density function (PDF) of the Weibull distribution is given by:
$$
f(x; k, \lambda) = \begin{cases} 
\frac{k}{\lambda} \left(\frac{x}{\lambda}\right)^{k-1} e^{-\left(\frac{x}{\lambda}\right)^k} & x \ge 0 \\
0 & x < 0  \end{cases}
$$

Where:
- $ k $ is the shape parameter.
- $ \lambda $ is the scale parameter.
- $ x $ is the variable.

## Parameter Estimation

We estimate the parameters  $k$ and $\lambda$ using curve fitting. The `curve_fit` function from the `scipy.optimize` module is used for this purpose. 

Given the data for gas consumption in 2023, we perform the following steps:

1. Estimate the shape and scale parameters using 2023 gas consumption.
2. Generate synthetic gas consumption data using the estimated parameters.

## Generating Synthetic Gas Consumption Data

We generate synthetic gas consumption data using the `weibull_min.rvs` function from the `scipy.stats` module. To simulate an increase in gas consumption in the middle of the dataset, we artificially increase the values of gas consumption at certain indices.

In [ ]:
# Set the seed for reproducibility
np.random.seed(seed=0)

# Estimate the parameters for the Weibull distribution
k_estimate = 1
scale_estimate = np.median(df[2023])

print(f'Estimated shape parameter (c): {k_estimate} and scale parameter: {scale_estimate}')

# Parameters for the Weibull distribution
shape = k_estimate  # Shape parameter
scale = scale_estimate  # Scale parameter
n = 100 # Number of rows in the dataset

# Generate gas consumption data using the Weibull distribution
gas_consumption = weibull_min.rvs(shape, scale=scale, size=n)

# Increase the values of gas consumption on the middle of the model
for i in range (int(n/2)-3, int(n/2)+3):
    gas_consumption[i] = gas_consumption[i] + (gas_consumption.max())

plot_index(gas_consumption)

# Adding Laplacian Noise for Differential Privacy

Differential privacy is a framework for quantifying the privacy guarantees provided when releasing statistical data. By adding noise to the data, we can mask individual data points, making it difficult to infer specific information about any individual in the dataset.

One common method for achieving differential privacy is to add Laplacian noise to the data. The Laplace distribution is defined by its scale parameter $b$, which is determined by the sensitivity of the function and the privacy budget $\epsilon$.

## Laplace Distribution

The probability density function (PDF) of the Laplace distribution is given by:

$$
f(x | \mu, b) = \frac{1}{2b} \exp\left(-\frac{|x - \mu|}{b}\right)
$$

Where:
- $\mu$ is the location parameter (mean).
- $b$ is the scale parameter.

The scale parameter $b$ is calculated as:

$$
b = \frac{\Delta f}{\epsilon}
$$

Where:
- $\Delta f$  is the sensitivity of the function.
- $\epsilon$  is the privacy budget.

### What is $\epsilon$-Differential Privacy?

$\epsilon$-Differential Privacy is a formal definition of privacy that quantifies how much information about an individual can be inferred from the output of a data analysis algorithm. It provides a mathematical guarantee that the presence or absence of a single individual’s data in the dataset does not significantly affect the output of the algorithm, thus ensuring privacy.

**Formally, $\epsilon$-Differential Privacy is defined as:**

For a randomized algorithm $A$, the definition of $\epsilon$-Differential Privacy is:

$$
\Pr[A(D) \in S] \leq \exp(\epsilon) \cdot \Pr[A(D') \in S]
$$

for all datasets $D$ and $D'$ that differ by only one individual's data, and for all possible outputs $S$ of the algorithm.

# Differential Privacy: Sensitivity and Epsilon

Differential privacy is a framework designed to ensure the privacy of individuals in a dataset while still allowing for useful statistical analyses. Two critical concepts in differential privacy are **sensitivity** and the **privacy budget ($\epsilon$)**. This section explains these concepts, their importance, and their application in adding Laplacian noise to data for privacy protection.

## Sensitivity and Epsilon

### 1. **Sensitivity**

**Sensitivity** measures the maximum amount by which a function’s output can change when a single individual's data is added or removed from the dataset. It reflects the highest potential divergence between two neighboring datasets that differ by just one individual's information.

For the **sum function**, sensitivity can be determined by the greatest possible impact that removing or adding a single data point has on the result. In our case, this sensitivity is the difference between the maximum and minimum values of the gas consumption data.

**Mathematically, sensitivity** for a sum function is given by:

$$
\text{Sensitivity} = \Delta f = \max_{D_1, D_2 \text{ differing in one record}} \| f(D_1) - f(D_2) \|
$$

Where:
- $D_1$ and $D_2$ are datasets that differ by only one individual's data.
- $f$ is the sum function applied to the dataset.
- $\|\cdot\|$ denotes the distance metric, typically the $L_1$ norm (absolute difference).

**Since we are calculating the sum of the gas consumption values**, the highest divergence is the difference between the maximum and minimum values:

`sensitivity = gas_consumption.max() - gas_consumption.min()`

### 2. **Privacy Budget ($\epsilon$)**
The parameter $\epsilon$ controls the level of privacy provided:

- **Smaller $\epsilon$** values imply stronger privacy guarantees but increase the amount of noise added to the data.
- **Larger $\epsilon$** values allow for less noise and higher data accuracy, but provide weaker privacy assurances.

**In our case, $\epsilon$ is set to 0.5**.


## Adding Laplacian Noise

To add Laplacian noise to the gas consumption data, we perform the following steps:

1. Define a function to generate Laplacian noise.
2. Calculate the sensitivity of the data.
3. Add Laplacian noise to the data points.
4. Ensure that the noisy values are non-negative.
`noisy_gas_consumption = gas_consumption + laplacian_noise`


In [ ]:
# Parameters
epsilon = 1# Privacy budget
sensitivity = gas_consumption.max() - gas_consumption.min()  # Sensitivity of the function

# Add Laplacian noise to the data points
laplacian_noise = gen_laplacian_noise(epsilon, sensitivity, size=n)
noisy_gas_consumption = gas_consumption + laplacian_noise

# Ensure that the noisy values are non-negative
noisy_gas_consumption = np.clip(noisy_gas_consumption, 0, None)
plot_index(noisy_gas_consumption, param='Noisy gas consumption data')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Laplacian distribution parameters
mu = 0  # mean of the distribution
b = sensitivity/epsilon  # scale (diversity) parameter

# Generate a range of values
x = np.linspace(-1e7, 1e7, 10000)

# Calculate the PDF of the Laplacian distribution
pdf = (1 / (2 * b)) * np.exp(-np.abs(x - mu) / b)

# Plot the PDF
plt.figure(figsize=(8, 6))
plt.plot(x, pdf)
plt.title('Probability Density Function of Laplacian Distribution')
plt.xlabel('x')
plt.ylabel('Probability Density')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plot_index(gas_consumption, noisy_gas_consumption)

## Generate syntetic distances based on Gaussian dist

In [ ]:
# Generate distance data using a normal distribution
distance_mean = 30
distance_std = 5
distances = np.random.normal(loc=distance_mean, scale=distance_std, size=n)

plt.figure(figsize=(8, 4))
plt.hist(distances, bins=30, edgecolor='k', alpha=0.7)
plt.title('Distance Distribution')
plt.xlabel('Distance')
plt.ylabel('Frequency')
plt.show()

## Calculating Heat Density and Setting the Limit

In this section, we calculate the **heat density** from the gas consumption data and use it to identify areas with high heat density. 

### What is Heat Density?

**Heat Density** measures the amount of heat consumption per unit distance.
The formula for calculating heat density is:

$$
\text{Heat Density} = \frac{\text{Gas Consumption}}{\text{Distance}}
$$

We also have set a `heat_density_limit` of 3000, meaning that if an area has a heat density higher than this, it is considered a valid point for a construction of a heat-grid.

In [ ]:
# Convert the data into a pandas DataFrame
df_generated = pd.DataFrame({
    'gas_consumption': gas_consumption,
    'distance': distances
})

noisy_df_generated = pd.DataFrame({
    'gas_consumption': noisy_gas_consumption,
    'distance': distances
})

df_generated['cumulative_distances'] = df_generated['distance'] / 2 + df_generated['distance'].cumsum().shift(fill_value=0)
noisy_df_generated['cumulative_distances'] = df_generated['cumulative_distances']

heat_density_limit = 5000

df_generated['heat_density'] = df_generated['gas_consumption'] / df_generated['distance']
noisy_df_generated['heat_density'] = noisy_df_generated['gas_consumption'] / noisy_df_generated['distance']
df_high_density = df_generated[df_generated['heat_density'] > heat_density_limit]
noisy_df_high_density = noisy_df_generated[noisy_df_generated['heat_density'] > heat_density_limit]

plot_limit(df_generated, noisy_df_generated, df_high_density, noisy_df_high_density, param='heat_density', close=False)

## Grouping Data Points and Calculating New Heat Densities

### How is New Heat Density Calculated?

To calculate the new heat density for grouped data points, we perform the following steps:

1. **Group the Data Points:**

   Data points are divided into groups of a specified size, called `group_size`. Within each group, we aggregate the `gas_consumption` and `distance` values.

2. **Sum of Gas Consumption and Distance:**

   For each group, we compute:
   - **Sum of Gas Consumption**: The total amount of gas used across all data points in the group.
   - **Sum of Distance**: The total distance covered by all data points in the group.

   This aggregated data allows us to analyze the heat density for the grouped data.

3. **Calculate Heat Density for Each Group:**

   To calculate the heat density for each group of data points of size `group_size`, we sum the gas consumption and distances over the range of data points in the group. The formula for calculating the heat density for each group is:

   $$
   \text{Heat Density}_i = \frac{\sum\limits_{j=i}^{i+\text{group size}-1} \text{Gas Consumption}_j}{\sum\limits_{j=i}^{i+\text{group size}-1} \text{Distance}_j}
   $$

   This formula is applied to the data in each group to determine the average heat density for that particular group.


In [ ]:
list_dfs = []
list_dfs_noisy = []
for i in range(1, n):
    df, noisy_df = calculate_grouped_heat_density(df_generated, epsilon, sensitivity, group_size=i)
    list_dfs.append(df)
    list_dfs_noisy.append(noisy_df)

In [ ]:
plot_distances(list_dfs[0][0:10], 'heat_density')
plot_distances(list_dfs[50][0:2], 'heat_density')

In [ ]:
heat_density_limit = 5000
df_high_density_list = []
noisy_df_high_density_list = []
plot_choice = 'heat_density' # 'heat_density' or 'gas_consumption'
for i in range(0, n-1):
    df_high_density = list_dfs[i][list_dfs[i]['heat_density'] > heat_density_limit]
    df_high_density_list.append(df_high_density)
    noisy_df_high_density = list_dfs_noisy[i][list_dfs_noisy[i]['heat_density'] > heat_density_limit]
    noisy_df_high_density_list.append(noisy_df_high_density)

    plot_limit(list_dfs[i], list_dfs_noisy[i], df_high_density, noisy_df_high_density, i, f'../images/heat_density/group_size_{i}.png', plot_choice)

## Find the Longest Consecutive Sequence:
   - Using the identified high-density indices, we find the longest sequence of consecutive indices. This sequence represents a contiguous region where the heat density is consistently high.
   - We apply this method to both the original and noisy datasets to compare and validate the results.

By focusing on the longest sequence of high-density indices, we can effectively determine the most suitable locations for constructing the heat grid.

In [ ]:
for i in range(n-1):
    high_density_indices = df_high_density_list[i].index
    noisy_high_density_indices = noisy_df_high_density_list[i].index

    longest_high_density_sequence = find_longest_consecutive_sequence(high_density_indices)
    longest_noisy_high_density_sequence = find_longest_consecutive_sequence(noisy_high_density_indices)

### Testing a Range of Epsilons for Differential Privacy

Trying to evaluate the impact of different values of the privacy budget ($\epsilon$), on the accuracy of our heat density calculations. The steps in our experiment were as follows:

1. **Fixed Group Size:**
   - We set the group size for aggregation to 5, meaning that we grouped the data points in sets of 5 to calculate the heat density.

2. **Range of Epsilons:**
   - We tested a range of $\epsilon$ values from $10^{-3}$ to $10^{4}$ using a logarithmic scale to cover a wide spectrum of privacy levels.

3. **Accuracy Measurements:**
   - For each $\epsilon$ value, we calculated the heat densities for both the original and noisy datasets.
   - We identified high-density indices in both datasets, where the heat density exceeded the predefined threshold.

4. **Jaccard Similarity:**
   - We measured the accuracy of the noisy dataset compared to the original dataset using the Jaccard similarity index. The Jaccard similarity is defined as:
     $$
     J(A, B) = \frac{|A \cap B|}{|A \cup B|}
     $$
     where $A$ and $B$ are the sets of high-density indices from the original and noisy datasets, respectively. The Jaccard similarity measures the similarity between the two sets, with a value of 1 indicating perfect agreement and 0 indicating no overlap.

5. **Comparison of Longest Sequences:**
   - We also checked whether the longest consecutive sequence of high-density indices was the same in both the original and noisy datasets. This step ensures that the locations for constructing the heat grid remain consistent even when noise is added for differential privacy.

In [ ]:
size = 5

list_dfs = []
list_dfs_noisy = []
epsilon_values = np.logspace(-3, 3, num=500)

accuracies = []
mutually_inclusive_results = []

for epsilon in epsilon_values:
    df, noisy_df = calculate_grouped_heat_density(df_generated, epsilon, sensitivity, group_size=size)
    df_high_density = df[df['heat_density'] > heat_density_limit]
    noisy_df_high_density = noisy_df[noisy_df['heat_density'] > heat_density_limit]
    list_dfs.append(df)
    list_dfs_noisy.append(noisy_df)

    # Assuming df and noisy_df can be compared directly as lists
    accuracy = jaccard_similarity(df_high_density.index, noisy_df_high_density.index)
    mutually_inclusive = equal_lists(find_longest_consecutive_sequence(df_high_density.index), find_longest_consecutive_sequence(noisy_df_high_density.index))

    accuracies.append(accuracy)
    mutually_inclusive_results.append(mutually_inclusive)

In [ ]:
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(epsilon_values, accuracies, marker='o', linestyle='-', color='b')
plt.xscale('log')
plt.xlabel('Epsilon')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy per Epsilon (Group Size = {})'.format(size))
plt.grid(True)

# Plotting mutually inclusive results per epsilon
plt.subplot(1, 2, 2)
plt.plot(epsilon_values, mutually_inclusive_results, marker='o', linestyle='-', color='r')
plt.xscale('log')
plt.xlabel('Epsilon')
plt.ylabel('Mutually Inclusive')
plt.title('Mutually Inclusive per Epsilon (Group Size = {})'.format(size))
plt.grid(True)
plt.yticks([0, 1], ['False', 'True'])

plt.tight_layout()
plt.show()